# Notebook 00 — Multi-Seed Runner

## What this does

Re-runs the full X-IDS pipeline at SEEDS = [123, 456, 789]. For each seed,
patches the 15 pipeline notebooks (using the verified-correct patch logic
from the dry-run notebook), executes them in order via `nbclient`, and
writes all outputs to seed-suffixed directories so the seed-42 baseline is
untouched.

## Compute budget

Expected wallclock: ~2.5-3 hours per seed. With resumability, can be spread
across 2-3 Colab Pro sessions.

## Two-phase execution

The notebook supports two modes via the `DRY_RUN_ONLY` switch in cell 2:

- `DRY_RUN_ONLY = True` (default): patches and saves notebooks, prints what
  WOULD run, but does NOT execute. Use this first to verify the runner
  state is sane.

- `DRY_RUN_ONLY = False`: actually executes everything. Set this only after
  the True run looks correct.

## Resumability

After each notebook completes, the runner writes a checkpoint file to
`models/seed_runs/seed{N}/_checkpoints/{notebook_name}.done`. On restart,
the runner skips notebooks whose checkpoints exist. To force a rerun, delete
the checkpoint file for that notebook.

If a notebook fails:
- Full traceback saved to `models/seed_runs/seed{N}/_errors/{notebook}.log`
- Partially-executed notebook saved to `models/seed_runs/seed{N}/_executed/{notebook}_failed.ipynb`
- Pipeline HALTS for that seed (does not run downstream notebooks)
- Other seeds continue if requested

## Safety guarantees

- Original notebooks on disk are NEVER modified
- Seed-42 outputs in `models/`, `calibrators/`, etc. are NEVER touched
- All seed-N outputs go to `**/seedN/` subdirectories
- Pre-existing partial seed-N outputs from a previous run are preserved
  (resumability based on checkpoint files, not on output presence)


In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, shutil
REPO = '/content/drive/MyDrive/XIDS_Research/xids-research'
os.chdir(REPO)

for f in ['.gitconfig', '.git-credentials']:
    src = f'/content/drive/MyDrive/XIDS_Research/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'/root/{f}')
        if f == '.git-credentials':
            os.chmod(f'/root/{f}', 0o600)

# Verify nbclient is available (should be on Colab by default)
try:
    import nbclient
    import nbformat
    print(f'nbclient: {nbclient.__version__}')
    print(f'nbformat: {nbformat.__version__}')
except ImportError as e:
    print(f'Install needed: !pip install nbclient nbformat')
    raise

print(f'Ready: {os.getcwd()}')

Mounted at /content/drive
nbclient: 0.10.4
nbformat: 5.10.4
Ready: /content/drive/MyDrive/XIDS_Research/xids-research


In [3]:
# ============================================================
# USER CONFIGURATION — edit these before running
# ============================================================

# DRY-RUN MODE: True = preview only, False = actually execute
DRY_RUN_ONLY = False

# Seeds to run (in order). Comment out seeds you don't want this session.
SEEDS_TO_RUN = [123, 456, 789, 2024, 31337]

# Per-notebook timeout in seconds (cell-level)
NOTEBOOK_TIMEOUT = 3600  # 1 hour per notebook (longer for SHAP)

# Stop seed-N execution after first failed notebook?
HALT_ON_FAILURE = True

# Continue to next seed if previous seed had a failure?
CONTINUE_NEXT_SEED_ON_FAILURE = True

# ============================================================
# (No editing below this line in normal use)
# ============================================================

print(f'Mode: {"DRY-RUN (no execution)" if DRY_RUN_ONLY else "FULL EXECUTION"}')
print(f'Seeds: {SEEDS_TO_RUN}')
print(f'Timeout per notebook: {NOTEBOOK_TIMEOUT}s ({NOTEBOOK_TIMEOUT // 60} min)')
print(f'Halt on failure within seed: {HALT_ON_FAILURE}')
print(f'Continue to next seed on failure: {CONTINUE_NEXT_SEED_ON_FAILURE}')

Mode: FULL EXECUTION
Seeds: [123, 456, 789, 2024, 31337]
Timeout per notebook: 3600s (60 min)
Halt on failure within seed: True
Continue to next seed on failure: True


In [4]:
import nbformat as nbf
from nbclient import NotebookClient
from nbclient.exceptions import CellExecutionError
import pandas as pd
import numpy as np
import re, copy, json, shutil, traceback
import time
from pathlib import Path
from datetime import datetime, timedelta

# Pipeline notebook list in execution order — identical to the dry-run notebook
PIPELINE = [
    ('02_train_models_v2.ipynb',           'nsl_kdd_v2',     'simple_seed'),
    ('02_unsw_train_models_v2.ipynb',      'unsw_nb15_v2',   'inject_seed_and_regex'),
    ('02_cic_train_models_v2.ipynb',       'cic_ids2017_v2', 'inject_seed_and_regex'),
    ('03_nsl_calibration_v2.ipynb',        'nsl_kdd_v2',     'simple_seed'),
    ('03_unsw_calibration_v2.ipynb',       'unsw_nb15_v2',   'simple_seed'),
    ('03_cic_calibration_v2.ipynb',        'cic_ids2017_v2', 'simple_seed'),
    ('03e_refit_hybrid_calibrators.ipynb', 'all',            'no_seed'),
    ('04c_shap_canonical.ipynb',           'all',            'simple_seed'),
    ('05c_stability_canonical.ipynb',      'all',            'simple_seed'),
    ('06_krishna_agreement_v3.ipynb',      'all',            'simple_seed'),
    ('07c_scts_canonical_mondrian_v2.ipynb','all',           'simple_seed'),
    ('07d_scts_calib_health.ipynb',        'all',            'no_seed'),
    ('07e_phase_a_strict_protocol.ipynb',  'all',            'simple_seed_keep_boot'),
    ('07f_phase_a_diagnostic.ipynb',       'all',            'simple_seed'),
    ('08_bootstrap_cis.ipynb',             'all',            'simple_seed_keep_defaults'),
]

print(f'Pipeline: {len(PIPELINE)} notebooks per seed')
print(f'Total executions planned: {len(PIPELINE) * len(SEEDS_TO_RUN)}')

Pipeline: 15 notebooks per seed
Total executions planned: 75


In [5]:
# Patch functions (identical to dry-run notebook — proven correct in verification)

def patch_simple_seed(source, new_seed):
    pattern = re.compile(r'^(\s*SEED\s*=\s*)42(\s*(?:#.*)?)$', re.MULTILINE)
    return pattern.sub(lambda m: f'{m.group(1)}{new_seed}{m.group(2)}', source)

def patch_inject_seed_and_regex(source, new_seed):
    if re.search(r'^\s*SEED\s*=\s*\d+', source, re.MULTILINE):
        source = patch_simple_seed(source, new_seed)
    return re.sub(r'random_state\s*=\s*42\b', 'random_state=SEED', source)

def patch_simple_seed_keep_boot(source, new_seed):
    return patch_simple_seed(source, new_seed)

def patch_simple_seed_keep_defaults(source, new_seed):
    return patch_simple_seed(source, new_seed)

def patch_no_seed(source, new_seed):
    return source

PATCH_FUNCTIONS = {
    'simple_seed': patch_simple_seed,
    'inject_seed_and_regex': patch_inject_seed_and_regex,
    'simple_seed_keep_boot': patch_simple_seed_keep_boot,
    'simple_seed_keep_defaults': patch_simple_seed_keep_defaults,
    'no_seed': patch_no_seed,
}

def build_path_override_cell(notebook_name, dataset_tag, new_seed):
    """Same as dry-run — returns Python source for the path-override cell."""
    lines = [
        f'# === MULTI-SEED PATCH: path overrides for seed={new_seed} ===',
        'from pathlib import Path as _P',
        f'_REPO = "{REPO}"',
        f'_SEED_TAG = "seed{new_seed}"',
        '',
        'try:',
        '    SEED',
        'except NameError:',
        f'    SEED = {new_seed}',
        '',
    ]
    if notebook_name == '02_train_models_v2.ipynb':
        lines.extend([
            'MODELS_DIR = _P(_REPO) / "models" / "nsl_kdd_v2" / _SEED_TAG',
            'PREDS_DIR = MODELS_DIR / "predictions"',
            'PROBS_DIR = MODELS_DIR / "probabilities"',
            'TABLES_DIR = _P(_REPO) / "results" / "tables" / _SEED_TAG',
            'for _d in [MODELS_DIR, PREDS_DIR, PROBS_DIR, TABLES_DIR]:',
            '    _d.mkdir(parents=True, exist_ok=True)',
        ])
    elif notebook_name == '02_unsw_train_models_v2.ipynb':
        lines.extend([
            'MODELS_DIR = _P(_REPO) / "models" / "unsw_nb15_v2" / _SEED_TAG',
            'PREDS_DIR = MODELS_DIR / "predictions"',
            'PROBS_DIR = MODELS_DIR / "probabilities"',
            'TABLES_DIR = _P(_REPO) / "results" / "tables" / _SEED_TAG',
            'for _d in [MODELS_DIR, PREDS_DIR, PROBS_DIR, TABLES_DIR]:',
            '    _d.mkdir(parents=True, exist_ok=True)',
        ])
    elif notebook_name == '02_cic_train_models_v2.ipynb':
        lines.extend([
            'MODELS_DIR = _P(_REPO) / "models" / "cic_ids2017_v2" / _SEED_TAG',
            'PREDS_DIR = MODELS_DIR / "predictions"',
            'PROBS_DIR = MODELS_DIR / "probabilities"',
            'TABLES_DIR = _P(_REPO) / "results" / "tables" / _SEED_TAG',
            'for _d in [MODELS_DIR, PREDS_DIR, PROBS_DIR, TABLES_DIR]:',
            '    _d.mkdir(parents=True, exist_ok=True)',
        ])
    elif notebook_name.startswith('03_') and 'calibration' in notebook_name:
        ds = dataset_tag
        lines.extend([
            f'CALIB_OUT_DIR = _P(_REPO) / "calibrators" / "{ds}" / _SEED_TAG',
            f'_MODELS_SEED_DIR = _P(_REPO) / "models" / "{ds}" / _SEED_TAG',
            'TABLES_DIR = _P(_REPO) / "results" / "tables" / _SEED_TAG',
            'CALIB_OUT_DIR.mkdir(parents=True, exist_ok=True)',
            'TABLES_DIR.mkdir(parents=True, exist_ok=True)',
            'try:',
            '    PROBS_DIR = _MODELS_SEED_DIR / "probabilities"',
            '    PREDS_DIR = _MODELS_SEED_DIR / "predictions"',
            'except Exception: pass',
        ])
    elif notebook_name == '03e_refit_hybrid_calibrators.ipynb':
        lines.extend([
            'for _ds in ["nsl_kdd_v2", "unsw_nb15_v2", "cic_ids2017_v2"]:',
            '    (_P(_REPO) / "calibrators" / _ds / _SEED_TAG).mkdir(parents=True, exist_ok=True)',
        ])
    elif notebook_name == '04c_shap_canonical.ipynb':
        lines.extend([
            'for _ds in ["nsl_kdd_v2", "unsw_nb15_v2", "cic_ids2017_v2"]:',
            '    (_P(_REPO) / "shap_values" / _ds / _SEED_TAG).mkdir(parents=True, exist_ok=True)',
            'TABLES_DIR = _P(_REPO) / "results" / "tables" / _SEED_TAG',
            'TABLES_DIR.mkdir(parents=True, exist_ok=True)',
        ])
    else:
        lines.extend([
            'TABLES_DIR = _P(_REPO) / "results" / "tables" / _SEED_TAG',
            'FIGURES_DIR = _P(_REPO) / "results" / "figures" / _SEED_TAG',
            'TABLES_DIR.mkdir(parents=True, exist_ok=True)',
            'FIGURES_DIR.mkdir(parents=True, exist_ok=True)',
            'for _ds in ["nsl_kdd_v2", "unsw_nb15_v2", "cic_ids2017_v2"]:',
            '    (_P(_REPO) / "calibrators" / _ds / _SEED_TAG).mkdir(parents=True, exist_ok=True)',
        ])
    lines.append('print(f"[multi-seed patch] SEED={SEED}, TABLES_DIR={TABLES_DIR}")')
    return '\n'.join(lines)

print('Patch functions defined.')

Patch functions defined.


In [6]:
def patch_notebook_for_seed(nb_path, notebook_name, dataset_tag, strategy, new_seed):
    """Load notebook, apply patches, return the in-memory NotebookNode."""
    nb_obj = nbf.read(str(nb_path), as_version=4)
    patch_fn = PATCH_FUNCTIONS[strategy]

    for cell in nb_obj.cells:
        if cell.cell_type == 'code':
            cell.source = patch_fn(cell.source, new_seed)

    # Inject override cell after the first imports/SEED cell
    override_code = build_path_override_cell(notebook_name, dataset_tag, new_seed)
    override_cell = nbf.v4.new_code_cell(override_code)

    insertion_idx = None
    for idx, cell in enumerate(nb_obj.cells):
        if cell.cell_type != 'code':
            continue
        if ('SEED' in cell.source) or ('import numpy' in cell.source) or ('import pandas' in cell.source):
            insertion_idx = idx + 1
            break
    if insertion_idx is None:
        for idx, cell in enumerate(nb_obj.cells):
            if cell.cell_type == 'code':
                insertion_idx = idx + 1
                break

    if insertion_idx is not None:
        nb_obj.cells.insert(insertion_idx, override_cell)

    return nb_obj, insertion_idx

print('Patcher ready.')

Patcher ready.


In [7]:
def get_seed_run_dirs(seed):
    base = Path(REPO) / 'models' / 'seed_runs' / f'seed{seed}'
    return {
        'base': base,
        'checkpoints': base / '_checkpoints',
        'executed': base / '_executed',
        'errors': base / '_errors',
        'logs': base / '_logs',
    }

def ensure_seed_dirs(seed):
    dirs = get_seed_run_dirs(seed)
    for d in dirs.values():
        d.mkdir(parents=True, exist_ok=True)
    return dirs

def is_checkpointed(seed, notebook_name):
    dirs = get_seed_run_dirs(seed)
    return (dirs['checkpoints'] / f'{notebook_name}.done').exists()

def write_checkpoint(seed, notebook_name, info):
    dirs = ensure_seed_dirs(seed)
    cp_path = dirs['checkpoints'] / f'{notebook_name}.done'
    with open(cp_path, 'w') as f:
        json.dump(info, f, indent=2, default=str)

def write_error_log(seed, notebook_name, traceback_str, extra=None):
    dirs = ensure_seed_dirs(seed)
    err_path = dirs['errors'] / f'{notebook_name}.log'
    with open(err_path, 'w') as f:
        f.write(f'Error at {datetime.now().isoformat()}\n')
        f.write(f'Notebook: {notebook_name}\n')
        f.write(f'Seed: {seed}\n')
        if extra:
            f.write(f'Extra info: {json.dumps(extra, default=str, indent=2)}\n')
        f.write(f'\n--- Traceback ---\n')
        f.write(traceback_str)
    return err_path

print('Checkpoint helpers ready.')

Checkpoint helpers ready.


In [8]:
def execute_patched_notebook(nb_obj, notebook_name, seed, timeout=NOTEBOOK_TIMEOUT):
    """Execute a patched in-memory notebook. Returns (success, error_info)."""
    client = NotebookClient(
        nb_obj,
        timeout=timeout,
        kernel_name='python3',
        resources={'metadata': {'path': REPO}},  # Run as if working dir is the repo
    )
    t_start = time.time()
    try:
        client.execute()
        elapsed = time.time() - t_start
        return True, {'elapsed_sec': elapsed, 'status': 'success'}
    except CellExecutionError as e:
        elapsed = time.time() - t_start
        tb = traceback.format_exc()
        return False, {'elapsed_sec': elapsed, 'status': 'cell_error',
                       'error_class': 'CellExecutionError', 'traceback': tb,
                       'message': str(e)[:500]}
    except Exception as e:
        elapsed = time.time() - t_start
        tb = traceback.format_exc()
        return False, {'elapsed_sec': elapsed, 'status': 'unexpected_error',
                       'error_class': type(e).__name__, 'traceback': tb,
                       'message': str(e)[:500]}

print('Executor ready.')

Executor ready.


In [9]:
import torch, shutil
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE"}')
total, used, free = shutil.disk_usage('/content/drive/MyDrive')
print(f'Drive free: {free / 1e9:.1f} GB')

CUDA available: True
GPU device: Tesla T4
Drive free: 192.4 GB


In [10]:
# === MAIN EXECUTION LOOP ===
nb_dir = Path(REPO) / 'notebooks'

print('=' * 80)
print(f'MULTI-SEED RUNNER — {datetime.now().strftime("%Y-%m-%d %H:%M")}')
print(f'Mode: {"DRY-RUN" if DRY_RUN_ONLY else "FULL EXECUTION"}')
print('=' * 80)

# Pre-flight: every pipeline notebook must exist
print('\nPre-flight check...')
all_present = True
for nb_name, _, _ in PIPELINE:
    p = nb_dir / nb_name
    status = 'OK' if p.exists() else 'MISSING'
    if not p.exists():
        all_present = False
        print(f'  [{status}] {nb_name}')
if all_present:
    print(f'  All {len(PIPELINE)} pipeline notebooks found.')
else:
    print('\nABORT: missing notebooks above. Fix paths before continuing.')
    raise SystemExit('Missing pipeline notebooks')

# Summary of work to do (after checkpoints)
print('\nWork to do:')
to_run_by_seed = {}
for seed in SEEDS_TO_RUN:
    to_run = []
    for nb_name, ds_tag, strategy in PIPELINE:
        if is_checkpointed(seed, nb_name):
            continue
        to_run.append((nb_name, ds_tag, strategy))
    to_run_by_seed[seed] = to_run
    print(f'  seed={seed}: {len(to_run)}/{len(PIPELINE)} notebooks to run')

total_to_run = sum(len(v) for v in to_run_by_seed.values())
print(f'\nTotal notebooks to execute: {total_to_run}')

if DRY_RUN_ONLY:
    print('\n--- DRY-RUN MODE: not executing anything ---')
    print('Set DRY_RUN_ONLY = False to actually run.')
    print('Patches will be applied in memory and saved to _executed/ but not run.')


MULTI-SEED RUNNER — 2026-06-09 06:24
Mode: FULL EXECUTION

Pre-flight check...
  All 15 pipeline notebooks found.

Work to do:
  seed=123: 15/15 notebooks to run
  seed=456: 15/15 notebooks to run
  seed=789: 15/15 notebooks to run
  seed=2024: 15/15 notebooks to run
  seed=31337: 15/15 notebooks to run

Total notebooks to execute: 75


In [11]:
# Execute per-seed, per-notebook
run_log = []
t0_global = time.time()

for seed in SEEDS_TO_RUN:
    dirs = ensure_seed_dirs(seed)
    seed_t_start = time.time()
    seed_failed = False

    print(f'\n{"="*80}')
    print(f'SEED = {seed}')
    print(f'{"="*80}')

    for nb_name, ds_tag, strategy in PIPELINE:
        # Skip if already done
        if is_checkpointed(seed, nb_name):
            print(f'  [SKIP] {nb_name} (checkpoint exists)')
            run_log.append({'seed': seed, 'notebook': nb_name, 'status': 'skipped',
                            'elapsed_sec': 0})
            continue

        # If a previous notebook in this seed failed and HALT_ON_FAILURE, stop
        if seed_failed and HALT_ON_FAILURE:
            print(f'  [HALT] {nb_name} (previous notebook failed, halting this seed)')
            run_log.append({'seed': seed, 'notebook': nb_name, 'status': 'halted',
                            'elapsed_sec': 0})
            continue

        # Patch
        src_path = nb_dir / nb_name
        nb_obj, insertion_idx = patch_notebook_for_seed(src_path, nb_name, ds_tag, strategy, seed)

        if DRY_RUN_ONLY:
            # Save patched notebook to _executed/ but don't run
            out_path = dirs['executed'] / nb_name
            with open(out_path, 'w') as f:
                nbf.write(nb_obj, f)
            print(f'  [DRY-RUN] {nb_name} (patched, saved to _executed/, would now execute)')
            run_log.append({'seed': seed, 'notebook': nb_name, 'status': 'dry_run',
                            'elapsed_sec': 0, 'patched_at_cell': insertion_idx})
            continue

        # REAL EXECUTION
        print(f'  [RUNNING] {nb_name} ...', end=' ', flush=True)
        t_nb_start = time.time()
        success, info = execute_patched_notebook(nb_obj, nb_name, seed)
        elapsed = info['elapsed_sec']

        # Save executed notebook (whether success or fail)
        if success:
            out_path = dirs['executed'] / nb_name
        else:
            out_path = dirs['executed'] / f'{nb_name.replace(".ipynb", "")}_failed.ipynb'
        with open(out_path, 'w') as f:
            nbf.write(nb_obj, f)

        if success:
            write_checkpoint(seed, nb_name, {
                'completed_at': datetime.now().isoformat(),
                'elapsed_sec': elapsed,
                'strategy': strategy,
                'override_inserted_at_cell': insertion_idx,
            })
            print(f'OK in {elapsed:.0f}s ({elapsed/60:.1f} min)')
            run_log.append({'seed': seed, 'notebook': nb_name, 'status': 'success',
                            'elapsed_sec': elapsed})
        else:
            err_path = write_error_log(seed, nb_name, info['traceback'], extra=info)
            print(f'FAILED ({info["error_class"]}) after {elapsed:.0f}s')
            print(f'    Message: {info["message"][:200]}')
            print(f'    Full log: {err_path}')
            print(f'    Failed notebook: {out_path}')
            seed_failed = True
            run_log.append({'seed': seed, 'notebook': nb_name, 'status': 'failed',
                            'elapsed_sec': elapsed, 'error': info['message'][:200]})

    seed_elapsed = time.time() - seed_t_start
    status = 'FAILED' if seed_failed else 'COMPLETE'
    print(f'\n  Seed {seed} {status} in {seed_elapsed:.0f}s ({seed_elapsed/60:.1f} min)')

    if seed_failed and not CONTINUE_NEXT_SEED_ON_FAILURE:
        print(f'  Halting all seeds (CONTINUE_NEXT_SEED_ON_FAILURE = False)')
        break

total_elapsed = time.time() - t0_global
print(f'\n{"="*80}')
print(f'Total elapsed: {total_elapsed:.0f}s ({total_elapsed/60:.1f} min, {total_elapsed/3600:.1f} h)')
print(f'{"="*80}')


SEED = 123
  [RUNNING] 02_train_models_v2.ipynb ... OK in 874s (14.6 min)
  [RUNNING] 02_unsw_train_models_v2.ipynb ... OK in 380s (6.3 min)
  [RUNNING] 02_cic_train_models_v2.ipynb ... OK in 476s (7.9 min)
  [RUNNING] 03_nsl_calibration_v2.ipynb ... FAILED (CellExecutionError) after 124s
    Message: An error occurred while executing the following cell:
------------------
ALL_MODELS = [
    ('rf_binary_cw',     'binary'),
    ('xgb_binary_cw',    'binary'),
    ('dnn_binary_cw',    'binary'),
    
    Full log: /content/drive/MyDrive/XIDS_Research/xids-research/models/seed_runs/seed123/_errors/03_nsl_calibration_v2.ipynb.log
    Failed notebook: /content/drive/MyDrive/XIDS_Research/xids-research/models/seed_runs/seed123/_executed/03_nsl_calibration_v2_failed.ipynb
  [HALT] 03_unsw_calibration_v2.ipynb (previous notebook failed, halting this seed)
  [HALT] 03_cic_calibration_v2.ipynb (previous notebook failed, halting this seed)
  [HALT] 03e_refit_hybrid_calibrators.ipynb (previous no


KeyboardInterrupt



In [1]:
from pathlib import Path
err_log = Path('/content/drive/MyDrive/XIDS_Research/xids-research/models/seed_runs/seed123/_errors/03_nsl_calibration_v2.ipynb.log')
print(err_log.read_text())

Error at 2026-06-09T06:55:39.076717
Notebook: 03_nsl_calibration_v2.ipynb
Seed: 123
Extra info: {
  "elapsed_sec": 123.81454801559448,
  "status": "cell_error",
  "error_class": "CellExecutionError",
  "traceback": "Traceback (most recent call last):\n  File \"/tmp/ipykernel_771/516752470.py\", line 11, in execute_patched_notebook\n    client.execute()\n  File \"/usr/local/lib/python3.12/dist-packages/jupyter_core/utils/__init__.py\", line 171, in wrapped\n    return _runner_map[name].run(inner)\n           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File \"/usr/local/lib/python3.12/dist-packages/jupyter_core/utils/__init__.py\", line 128, in run\n    return fut.result(None)\n           ^^^^^^^^^^^^^^^^\n  File \"/usr/lib/python3.12/concurrent/futures/_base.py\", line 456, in result\n    return self.__get_result()\n           ^^^^^^^^^^^^^^^^^^^\n  File \"/usr/lib/python3.12/concurrent/futures/_base.py\", line 401, in __get_result\n    raise self._exception\n  File \"/usr/local/lib/python3.12/dist

In [2]:
from pathlib import Path

REPO = '/content/drive/MyDrive/XIDS_Research/xids-research'

# What did seed-42 produce?
seed42_dir = Path(REPO) / 'models' / 'nsl_kdd_v2' / 'predictions'
print('seed=42 predictions files (NSL):')
for f in sorted(seed42_dir.glob('*.npy'))[:20]:
    print(f'  {f.name}')

print()

# What did seed-123 produce?
seed123_dir = Path(REPO) / 'models' / 'nsl_kdd_v2' / 'seed123' / 'predictions'
print('seed=123 predictions files (NSL):')
if seed123_dir.exists():
    for f in sorted(seed123_dir.glob('*.npy'))[:20]:
        print(f'  {f.name}')
else:
    print('  DIRECTORY DOES NOT EXIST')

# Also check what 02_train_models_v2 produces by scanning the original notebook
import nbformat as nbf
nb = nbf.read(str(Path(REPO) / 'notebooks' / '02_train_models_v2.ipynb'), as_version=4)
text = '\n'.join(c.source for c in nb.cells if c.cell_type == 'code')

# Find what model names get trained
import re
# Look for tuples like ('rf_binary_cw', ...) etc.
trained = set()
for m in re.finditer(r"\(\s*'(\w+)'\s*,", text):
    name = m.group(1)
    if 'binary' in name or '5class' in name:
        trained.add(name)
print()
print(f'Models that 02_train_models_v2.ipynb references in code: {sorted(trained)}')

seed=42 predictions files (NSL):
  dnn_5class_cw_calib_pred.npy
  dnn_5class_cw_calib_proba.npy
  dnn_5class_cw_test_pred.npy
  dnn_5class_cw_test_proba.npy
  dnn_5class_smote_calib_pred.npy
  dnn_5class_smote_calib_proba.npy
  dnn_5class_smote_test_pred.npy
  dnn_5class_smote_test_proba.npy
  dnn_binary_cw_calib_pred.npy
  dnn_binary_cw_calib_proba.npy
  dnn_binary_cw_test_pred.npy
  dnn_binary_cw_test_proba.npy
  rf_5class_cw_calib_pred.npy
  rf_5class_cw_calib_proba.npy
  rf_5class_cw_test_pred.npy
  rf_5class_cw_test_proba.npy
  rf_5class_smote_calib_pred.npy
  rf_5class_smote_calib_proba.npy
  rf_5class_smote_test_pred.npy
  rf_5class_smote_test_proba.npy

seed=123 predictions files (NSL):

Models that 02_train_models_v2.ipynb references in code: ['dnn_5class_cw', 'dnn_5class_smote', 'dnn_binary_cw', 'rf_5class_cw', 'rf_5class_smote', 'rf_binary_cw', 'xgb_5class_cw', 'xgb_5class_smote', 'xgb_binary_cw']


In [3]:
from pathlib import Path
import subprocess

REPO = '/content/drive/MyDrive/XIDS_Research/xids-research'

# Check all possible seed-123 NSL output locations
candidates = [
    Path(REPO) / 'models' / 'nsl_kdd_v2' / 'seed123',
    Path(REPO) / 'models' / 'nsl_kdd_v2' / 'seed123' / 'predictions',
    Path(REPO) / 'models' / 'nsl_kdd_v2' / 'seed123' / 'probabilities',
    Path(REPO) / 'models' / 'nsl_kdd_v2' / 'predictions',  # main dir — should NOT have seed-123 files
]

for d in candidates:
    print(f'\n{d}')
    if d.exists():
        files = sorted(d.iterdir())
        print(f'  ({len(files)} items)')
        for f in files[:30]:
            size = f.stat().st_size if f.is_file() else 'DIR'
            print(f'  {f.name} ({size} bytes)' if f.is_file() else f'  {f.name}/')
    else:
        print('  NOT EXIST')

# Also: where did the patched notebook think MODELS_DIR was?
# Re-read the patched notebook from _executed and check the override cell
import nbformat as nbf
exec_nb = Path(REPO) / 'models' / 'seed_runs' / 'seed123' / '_executed' / '02_train_models_v2.ipynb'
if exec_nb.exists():
    nb = nbf.read(str(exec_nb), as_version=4)
    print(f'\n\n--- Override cell content from executed 02_train_models_v2 ---')
    for c in nb.cells:
        if c.cell_type == 'code' and 'MULTI-SEED PATCH' in c.source:
            print(c.source)
            break

# Glob search the whole models dir tree for any seed123 NSL files
print(f'\n\n--- All NSL seed123 files anywhere under models/ ---')
import os
for root, dirs, files in os.walk(Path(REPO) / 'models' / 'nsl_kdd_v2'):
    for f in files:
        full = Path(root) / f
        if 'seed123' in str(full) or ('seed' in str(full).lower() and '123' in str(full)):
            print(f'  {full}')


/content/drive/MyDrive/XIDS_Research/xids-research/models/nsl_kdd_v2/seed123
  (2 items)
  predictions/
  probabilities/

/content/drive/MyDrive/XIDS_Research/xids-research/models/nsl_kdd_v2/seed123/predictions
  (0 items)

/content/drive/MyDrive/XIDS_Research/xids-research/models/nsl_kdd_v2/seed123/probabilities
  (0 items)

/content/drive/MyDrive/XIDS_Research/xids-research/models/nsl_kdd_v2/predictions
  (36 items)
  dnn_5class_cw_calib_pred.npy (201688 bytes)
  dnn_5class_cw_calib_proba.npy (504028 bytes)
  dnn_5class_cw_test_pred.npy (180480 bytes)
  dnn_5class_cw_test_proba.npy (451008 bytes)
  dnn_5class_smote_calib_pred.npy (201688 bytes)
  dnn_5class_smote_calib_proba.npy (504028 bytes)
  dnn_5class_smote_test_pred.npy (180480 bytes)
  dnn_5class_smote_test_proba.npy (451008 bytes)
  dnn_binary_cw_calib_pred.npy (201688 bytes)
  dnn_binary_cw_calib_proba.npy (201688 bytes)
  dnn_binary_cw_test_pred.npy (180480 bytes)
  dnn_binary_cw_test_proba.npy (180480 bytes)
  rf_5class_c

In [4]:
import os
from pathlib import Path
from datetime import datetime

REPO = '/content/drive/MyDrive/XIDS_Research/xids-research'
preds_dir = Path(REPO) / 'models' / 'nsl_kdd_v2' / 'predictions'

# Check modification timestamps of files in the seed-42 location
print('Recent file timestamps in models/nsl_kdd_v2/predictions/ (seed-42 location):')
files_by_time = sorted(preds_dir.iterdir(), key=lambda f: f.stat().st_mtime, reverse=True)
for f in files_by_time[:20]:
    mtime = datetime.fromtimestamp(f.stat().st_mtime)
    print(f'  {mtime.strftime("%Y-%m-%d %H:%M:%S")}  {f.name}')

print()
print('Current time:', datetime.now())

Recent file timestamps in models/nsl_kdd_v2/predictions/ (seed-42 location):
  2026-06-09 07:26:05  xgb_5class_cw_test_pred.npy
  2026-06-09 07:26:05  xgb_5class_cw_test_proba.npy
  2026-06-09 07:26:05  xgb_5class_cw_calib_pred.npy
  2026-06-09 07:26:05  xgb_5class_cw_calib_proba.npy
  2026-06-09 07:25:48  rf_5class_cw_calib_pred.npy
  2026-06-09 07:25:48  rf_5class_cw_test_proba.npy
  2026-06-09 07:25:48  rf_5class_cw_calib_proba.npy
  2026-06-09 07:25:48  rf_5class_cw_test_pred.npy
  2026-06-09 07:25:43  dnn_5class_smote_test_pred.npy
  2026-06-09 07:25:43  dnn_5class_smote_test_proba.npy
  2026-06-09 07:25:43  dnn_5class_smote_calib_pred.npy
  2026-06-09 07:25:43  dnn_5class_smote_calib_proba.npy
  2026-06-09 07:23:23  xgb_5class_smote_test_pred.npy
  2026-06-09 07:23:23  xgb_5class_smote_test_proba.npy
  2026-06-09 07:23:23  xgb_5class_smote_calib_pred.npy
  2026-06-09 07:23:23  xgb_5class_smote_calib_proba.npy
  2026-06-09 07:22:39  rf_5class_smote_calib_proba.npy
  2026-06-09 07:

In [5]:
print('If you see this, the kernel is alive but not running cell 9.')
print('Check the Colab UI: cell 9 should show no spinner, no "running" indicator.')

If you see this, the kernel is alive but not running cell 9.
Check the Colab UI: cell 9 should show no spinner, no "running" indicator.


In [6]:
import os
os.chdir('/content/drive/MyDrive/XIDS_Research/xids-research')

# Check if model files are tracked in git
print('=== git log for dnn_5class_cw_test_proba.npy ===')
!git log --oneline -5 -- models/nsl_kdd_v2/predictions/dnn_5class_cw_test_proba.npy 2>&1 | head -20

print()
print('=== git ls-files check (is the file tracked?) ===')
!git ls-files models/nsl_kdd_v2/predictions/dnn_5class_cw_test_proba.npy 2>&1

print()
print('=== .gitignore ===')
!cat .gitignore 2>/dev/null

print()
print('=== git status models/nsl_kdd_v2/predictions/ ===')
!git status models/nsl_kdd_v2/predictions/ 2>&1 | head -30

=== git log for dnn_5class_cw_test_proba.npy ===

=== git ls-files check (is the file tracked?) ===

=== .gitignore ===
# Byte-compiled / optimized / DLL files
__pycache__/
*.py[codz]
*$py.class

# C extensions
*.so

# Distribution / packaging
.Python
build/
develop-eggs/
dist/
downloads/
eggs/
.eggs/
lib/
lib64/
parts/
sdist/
var/
wheels/
share/python-wheels/
*.egg-info/
.installed.cfg
*.egg
MANIFEST

# PyInstaller
#   Usually these files are written by a python script from a template
#   before PyInstaller builds the exe, so as to inject date/other infos into it.
*.manifest
*.spec

# Installer logs
pip-log.txt
pip-delete-this-directory.txt

# Unit test / coverage reports
htmlcov/
.tox/
.nox/
.coverage
.coverage.*
.cache
nosetests.xml
coverage.xml
*.cover
*.py.cover
.hypothesis/
.pytest_cache/
cover/

# Translations
*.mo
*.pot

# Django stuff:
*.log
local_settings.py
db.sqlite3
db.sqlite3-journal

# Flask stuff:
instance/
.webassets-cache

# Scrapy stuff:
.scrapy

# Sphinx documentati

In [7]:
import os
from pathlib import Path
from datetime import datetime

REPO = '/content/drive/MyDrive/XIDS_Research/xids-research'

for ds in ['nsl_kdd_v2', 'unsw_nb15_v2', 'cic_ids2017_v2']:
    print(f'\n=== {ds} ===')
    # Check several common subdirs
    for subdir in ['predictions', 'probabilities']:
        d = Path(REPO) / 'models' / ds / subdir
        if not d.exists():
            print(f'  {subdir}/: does not exist')
            continue
        files = sorted(d.iterdir(), key=lambda f: f.stat().st_mtime, reverse=True)
        print(f'  {subdir}/ ({len(files)} files)')
        if files:
            newest = files[0]
            oldest = files[-1]
            mt_new = datetime.fromtimestamp(newest.stat().st_mtime)
            mt_old = datetime.fromtimestamp(oldest.stat().st_mtime)
            print(f'    Newest: {mt_new}  {newest.name}')
            print(f'    Oldest: {mt_old}  {oldest.name}')


=== nsl_kdd_v2 ===
  predictions/ (36 files)
    Newest: 2026-06-09 07:26:05  xgb_5class_cw_test_pred.npy
    Oldest: 2026-06-09 07:03:31  dnn_5class_cw_calib_proba.npy
  probabilities/: does not exist

=== unsw_nb15_v2 ===
  predictions/ (18 files)
    Newest: 2026-05-28 00:49:59  dnn_5class_cw_test_pred.npy
    Oldest: 2026-05-28 00:43:38  rf_binary_cw_calib_pred.npy
  probabilities/ (18 files)
    Newest: 2026-05-28 00:49:59  dnn_5class_cw_test_proba.npy
    Oldest: 2026-05-28 00:43:38  rf_binary_cw_test_proba.npy

=== cic_ids2017_v2 ===
  predictions/ (18 files)
    Newest: 2026-05-28 01:14:09  dnn_5class_cw_test_pred.npy
    Oldest: 2026-05-28 01:05:31  rf_binary_cw_calib_pred.npy
  probabilities/ (18 files)
    Newest: 2026-05-28 01:14:09  dnn_5class_cw_test_proba.npy
    Oldest: 2026-05-28 01:05:31  rf_binary_cw_calib_proba.npy


In [8]:
import nbformat as nbf
from pathlib import Path
import re

REPO = '/content/drive/MyDrive/XIDS_Research/xids-research'

for nb_name in ['02_train_models_v2.ipynb', '02_unsw_train_models_v2.ipynb', '02_cic_train_models_v2.ipynb']:
    p = Path(REPO) / 'notebooks' / nb_name
    nb = nbf.read(str(p), as_version=4)
    print(f'\n=== {nb_name} ===')
    print('Cells defining paths:')
    for ci, c in enumerate(nb.cells):
        if c.cell_type != 'code': continue
        for li, line in enumerate(c.source.split('\n')):
            # Look for path-defining lines
            if re.search(r'^\s*(MODELS_DIR|PREDS_DIR|PROBS_DIR|TABLES_DIR)\s*=', line):
                print(f'  cell {ci} line {li}: {line.strip()[:140]}')


=== 02_train_models_v2.ipynb ===
Cells defining paths:
  cell 4 line 2: MODELS_DIR = Path(REPO) / 'models' / 'nsl_kdd_v2'
  cell 4 line 3: PREDS_DIR = MODELS_DIR / 'predictions'
  cell 4 line 4: TABLES_DIR = Path(REPO) / 'results' / 'tables'

=== 02_unsw_train_models_v2.ipynb ===
Cells defining paths:
  cell 5 line 8: MODELS_DIR = REPO / 'models/unsw_nb15_v2'
  cell 5 line 9: PREDS_DIR  = MODELS_DIR / 'predictions'
  cell 5 line 10: PROBS_DIR  = MODELS_DIR / 'probabilities'
  cell 5 line 11: TABLES_DIR = REPO / 'results/tables'

=== 02_cic_train_models_v2.ipynb ===
Cells defining paths:
  cell 5 line 8: MODELS_DIR = REPO_P / 'models/cic_ids2017_v2'
  cell 5 line 9: PREDS_DIR = MODELS_DIR / 'predictions'
  cell 5 line 10: PROBS_DIR = MODELS_DIR / 'probabilities'
  cell 5 line 11: TABLES_DIR = REPO_P / 'results/tables'


In [9]:
import os, shutil
from pathlib import Path
from datetime import datetime

REPO = '/content/drive/MyDrive/XIDS_Research/xids-research'

# Step A: list what's in the contaminated NSL predictions dir
src_dir = Path(REPO) / 'models' / 'nsl_kdd_v2' / 'predictions'
print(f'Current contents of {src_dir} ({len(list(src_dir.iterdir()))} items):')
for f in sorted(src_dir.iterdir()):
    mt = datetime.fromtimestamp(f.stat().st_mtime)
    print(f'  {mt.strftime("%Y-%m-%d %H:%M:%S")}  {f.name}')

# Step B: identify the 9 OLD binary model files (if they survived) vs the freshly-written 5class ones
# A binary file from seed-42 would have an old timestamp; if it's from today it's contaminated too
print('\n--- separating files by age ---')
files_today = []
files_old = []
for f in src_dir.iterdir():
    mt = f.stat().st_mtime
    if mt > 1717000000:  # epoch ~ June 2026 (anything from "today")
        # better check: today's date
        if datetime.fromtimestamp(mt).date() == datetime.now().date():
            files_today.append(f)
        else:
            files_old.append(f)
    else:
        files_old.append(f)

print(f'Files modified TODAY (contaminated/new from seed-456): {len(files_today)}')
for f in files_today:
    print(f'  {f.name}')
print(f'\nFiles from BEFORE today (surviving seed-42): {len(files_old)}')
for f in files_old:
    mt = datetime.fromtimestamp(f.stat().st_mtime)
    print(f'  {mt.strftime("%Y-%m-%d")}  {f.name}')

# Step C: prepare destination
dst_dir = Path(REPO) / 'models' / 'nsl_kdd_v2' / 'seed456' / 'predictions'
dst_dir.mkdir(parents=True, exist_ok=True)
print(f'\nDestination for seed-456 outputs: {dst_dir}')
print(f'Currently has {len(list(dst_dir.iterdir()))} files')

print('\n*** This cell only INSPECTS. No files moved yet. Confirm and I will write the move cell. ***')

Current contents of /content/drive/MyDrive/XIDS_Research/xids-research/models/nsl_kdd_v2/predictions (36 items):
  2026-06-09 07:03:31  dnn_5class_cw_calib_pred.npy
  2026-06-09 07:03:31  dnn_5class_cw_calib_proba.npy
  2026-06-09 07:03:31  dnn_5class_cw_test_pred.npy
  2026-06-09 07:03:31  dnn_5class_cw_test_proba.npy
  2026-06-09 07:25:43  dnn_5class_smote_calib_pred.npy
  2026-06-09 07:25:43  dnn_5class_smote_calib_proba.npy
  2026-06-09 07:25:43  dnn_5class_smote_test_pred.npy
  2026-06-09 07:25:43  dnn_5class_smote_test_proba.npy
  2026-06-09 07:22:15  dnn_binary_cw_calib_pred.npy
  2026-06-09 07:22:15  dnn_binary_cw_calib_proba.npy
  2026-06-09 07:22:15  dnn_binary_cw_test_pred.npy
  2026-06-09 07:22:15  dnn_binary_cw_test_proba.npy
  2026-06-09 07:25:48  rf_5class_cw_calib_pred.npy
  2026-06-09 07:25:48  rf_5class_cw_calib_proba.npy
  2026-06-09 07:25:48  rf_5class_cw_test_pred.npy
  2026-06-09 07:25:48  rf_5class_cw_test_proba.npy
  2026-06-09 07:22:39  rf_5class_smote_calib_pr

In [10]:
import os
from pathlib import Path
from datetime import datetime

REPO = '/content/drive/MyDrive/XIDS_Research/xids-research'

# Sanity check: only delete files from today
src_dir = Path(REPO) / 'models' / 'nsl_kdd_v2' / 'predictions'
today = datetime.now().date()

to_delete = []
for f in src_dir.iterdir():
    if datetime.fromtimestamp(f.stat().st_mtime).date() == today:
        to_delete.append(f)

print(f'About to delete {len(to_delete)} files modified today from {src_dir}')
print(f'UNSW and CIC directories will be untouched.')
print()

# Show what we're deleting
for f in sorted(to_delete):
    mt = datetime.fromtimestamp(f.stat().st_mtime)
    size = f.stat().st_size
    print(f'  DELETE: {f.name} ({size:,} bytes, modified {mt.strftime("%H:%M:%S")})')

# Confirm before delete
print()
print(f'Proceeding to delete in 3 seconds... interrupt cell if you want to abort')
import time
time.sleep(3)

n_deleted = 0
for f in to_delete:
    f.unlink()
    n_deleted += 1

print(f'\nDeleted {n_deleted} files.')

# Confirm directory is now empty (or only has surviving seed-42 files)
remaining = list(src_dir.iterdir())
print(f'Remaining files in {src_dir}: {len(remaining)}')
for f in sorted(remaining):
    mt = datetime.fromtimestamp(f.stat().st_mtime)
    print(f'  {mt.strftime("%Y-%m-%d %H:%M:%S")}  {f.name}')

# Also clean up the empty seed-123 and seed-456 directories
for s in [123, 456]:
    seed_dir = Path(REPO) / 'models' / 'nsl_kdd_v2' / f'seed{s}'
    if seed_dir.exists():
        # Remove the empty subdirs
        import shutil
        shutil.rmtree(seed_dir)
        print(f'Cleaned up: {seed_dir}')

print()
print('Cleanup complete. Ready for fix + retrain.')

About to delete 36 files modified today from /content/drive/MyDrive/XIDS_Research/xids-research/models/nsl_kdd_v2/predictions
UNSW and CIC directories will be untouched.

  DELETE: dnn_5class_cw_calib_pred.npy (201,688 bytes, modified 07:03:31)
  DELETE: dnn_5class_cw_calib_proba.npy (504,028 bytes, modified 07:03:31)
  DELETE: dnn_5class_cw_test_pred.npy (180,480 bytes, modified 07:03:31)
  DELETE: dnn_5class_cw_test_proba.npy (451,008 bytes, modified 07:03:31)
  DELETE: dnn_5class_smote_calib_pred.npy (201,688 bytes, modified 07:25:43)
  DELETE: dnn_5class_smote_calib_proba.npy (504,028 bytes, modified 07:25:43)
  DELETE: dnn_5class_smote_test_pred.npy (180,480 bytes, modified 07:25:43)
  DELETE: dnn_5class_smote_test_proba.npy (451,008 bytes, modified 07:25:43)
  DELETE: dnn_binary_cw_calib_pred.npy (201,688 bytes, modified 07:22:15)
  DELETE: dnn_binary_cw_calib_proba.npy (201,688 bytes, modified 07:22:15)
  DELETE: dnn_binary_cw_test_pred.npy (180,480 bytes, modified 07:22:15)
  D

In [11]:
# === FIXED INJECTION LOGIC ===
# Bug: original logic inserted override AFTER first cell with SEED/imports.
# For NSL training, SEED and MODELS_DIR are in adjacent cells, so override
# was inserted BEFORE the MODELS_DIR-defining cell, and the notebook then
# clobbered our paths.
#
# Fix: insert override AFTER the LAST cell defining path constants
# (MODELS_DIR, PREDS_DIR, PROBS_DIR, TABLES_DIR, CALIB_OUT_DIR).

import re

def patch_notebook_for_seed(nb_path, notebook_name, dataset_tag, strategy, new_seed):
    nb_obj = nbf.read(str(nb_path), as_version=4)
    patch_fn = PATCH_FUNCTIONS[strategy]

    for cell in nb_obj.cells:
        if cell.cell_type == 'code':
            cell.source = patch_fn(cell.source, new_seed)

    override_code = build_path_override_cell(notebook_name, dataset_tag, new_seed)
    override_cell = nbf.v4.new_code_cell(override_code)

    # NEW: Find the LAST code cell defining path constants
    path_pattern = re.compile(
        r'^\s*(MODELS_DIR|PREDS_DIR|PROBS_DIR|TABLES_DIR|CALIB_OUT_DIR|FIGURES_DIR|SHAP_DIR)\s*=',
        re.MULTILINE
    )
    insertion_idx = None
    for idx, cell in enumerate(nb_obj.cells):
        if cell.cell_type != 'code':
            continue
        if path_pattern.search(cell.source):
            insertion_idx = idx + 1  # Insert AFTER this cell

    # Fallback: if no path-defining cell found, use old logic
    if insertion_idx is None:
        for idx, cell in enumerate(nb_obj.cells):
            if cell.cell_type != 'code':
                continue
            if ('SEED' in cell.source) or ('import numpy' in cell.source) or ('import pandas' in cell.source):
                insertion_idx = idx + 1
                break
    if insertion_idx is None:
        for idx, cell in enumerate(nb_obj.cells):
            if cell.cell_type == 'code':
                insertion_idx = idx + 1
                break

    if insertion_idx is not None:
        nb_obj.cells.insert(insertion_idx, override_cell)

    return nb_obj, insertion_idx

print('Fixed patch_notebook_for_seed function defined.')
print('New logic: override cell inserted AFTER last cell defining path constants.')

Fixed patch_notebook_for_seed function defined.
New logic: override cell inserted AFTER last cell defining path constants.


In [13]:
# Re-define everything needed for verification — no dependency on previous kernel state
import nbformat as nbf
import re
from pathlib import Path

REPO = '/content/drive/MyDrive/XIDS_Research/xids-research'

PIPELINE = [
    ('02_train_models_v2.ipynb',           'nsl_kdd_v2',     'simple_seed'),
    ('02_unsw_train_models_v2.ipynb',      'unsw_nb15_v2',   'inject_seed_and_regex'),
    ('02_cic_train_models_v2.ipynb',       'cic_ids2017_v2', 'inject_seed_and_regex'),
    ('03_nsl_calibration_v2.ipynb',        'nsl_kdd_v2',     'simple_seed'),
    ('03_unsw_calibration_v2.ipynb',       'unsw_nb15_v2',   'simple_seed'),
    ('03_cic_calibration_v2.ipynb',        'cic_ids2017_v2', 'simple_seed'),
    ('03e_refit_hybrid_calibrators.ipynb', 'all',            'no_seed'),
    ('04c_shap_canonical.ipynb',           'all',            'simple_seed'),
    ('05c_stability_canonical.ipynb',      'all',            'simple_seed'),
    ('06_krishna_agreement_v3.ipynb',      'all',            'simple_seed'),
    ('07c_scts_canonical_mondrian_v2.ipynb','all',           'simple_seed'),
    ('07d_scts_calib_health.ipynb',        'all',            'no_seed'),
    ('07e_phase_a_strict_protocol.ipynb',  'all',            'simple_seed_keep_boot'),
    ('07f_phase_a_diagnostic.ipynb',       'all',            'simple_seed'),
    ('08_bootstrap_cis.ipynb',             'all',            'simple_seed_keep_defaults'),
]

# Patch primitives
def patch_simple_seed(source, new_seed):
    pattern = re.compile(r'^(\s*SEED\s*=\s*)42(\s*(?:#.*)?)$', re.MULTILINE)
    return pattern.sub(lambda m: f'{m.group(1)}{new_seed}{m.group(2)}', source)

def patch_inject_seed_and_regex(source, new_seed):
    if re.search(r'^\s*SEED\s*=\s*\d+', source, re.MULTILINE):
        source = patch_simple_seed(source, new_seed)
    return re.sub(r'random_state\s*=\s*42\b', 'random_state=SEED', source)

def patch_no_seed(source, new_seed):
    return source

PATCH_FUNCTIONS = {
    'simple_seed': patch_simple_seed,
    'inject_seed_and_regex': patch_inject_seed_and_regex,
    'simple_seed_keep_boot': patch_simple_seed,
    'simple_seed_keep_defaults': patch_simple_seed,
    'no_seed': patch_no_seed,
}

def build_path_override_cell(notebook_name, dataset_tag, new_seed):
    """Returns Python source for the path-override cell."""
    # SPECIAL: seed=42 uses original (non-suffixed) paths so retraining seed-42 baseline
    # writes back to the original location
    if new_seed == 42:
        seed_tag_assign = '_SEED_TAG = ""'
        seed_subdir_for_models = ''
        seed_subdir_for_tables = ''
    else:
        seed_tag_assign = f'_SEED_TAG = "seed{new_seed}"'
        seed_subdir_for_models = ' / _SEED_TAG'
        seed_subdir_for_tables = ' / _SEED_TAG'

    lines = [
        f'# === MULTI-SEED PATCH: path overrides for seed={new_seed} ===',
        'from pathlib import Path as _P',
        f'_REPO = "{REPO}"',
        seed_tag_assign,
        '',
        'try:',
        '    SEED',
        'except NameError:',
        f'    SEED = {new_seed}',
        '',
    ]

    if notebook_name == '02_train_models_v2.ipynb':
        lines.extend([
            f'MODELS_DIR = _P(_REPO) / "models" / "nsl_kdd_v2"{seed_subdir_for_models}',
            'PREDS_DIR = MODELS_DIR / "predictions"',
            'PROBS_DIR = MODELS_DIR / "probabilities"',
            f'TABLES_DIR = _P(_REPO) / "results" / "tables"{seed_subdir_for_tables}',
            'for _d in [MODELS_DIR, PREDS_DIR, PROBS_DIR, TABLES_DIR]:',
            '    _d.mkdir(parents=True, exist_ok=True)',
        ])
    elif notebook_name == '02_unsw_train_models_v2.ipynb':
        lines.extend([
            f'MODELS_DIR = _P(_REPO) / "models" / "unsw_nb15_v2"{seed_subdir_for_models}',
            'PREDS_DIR = MODELS_DIR / "predictions"',
            'PROBS_DIR = MODELS_DIR / "probabilities"',
            f'TABLES_DIR = _P(_REPO) / "results" / "tables"{seed_subdir_for_tables}',
            'for _d in [MODELS_DIR, PREDS_DIR, PROBS_DIR, TABLES_DIR]:',
            '    _d.mkdir(parents=True, exist_ok=True)',
        ])
    elif notebook_name == '02_cic_train_models_v2.ipynb':
        lines.extend([
            f'MODELS_DIR = _P(_REPO) / "models" / "cic_ids2017_v2"{seed_subdir_for_models}',
            'PREDS_DIR = MODELS_DIR / "predictions"',
            'PROBS_DIR = MODELS_DIR / "probabilities"',
            f'TABLES_DIR = _P(_REPO) / "results" / "tables"{seed_subdir_for_tables}',
            'for _d in [MODELS_DIR, PREDS_DIR, PROBS_DIR, TABLES_DIR]:',
            '    _d.mkdir(parents=True, exist_ok=True)',
        ])
    elif notebook_name.startswith('03_') and 'calibration' in notebook_name:
        ds = dataset_tag
        lines.extend([
            f'CALIB_OUT_DIR = _P(_REPO) / "calibrators" / "{ds}"{seed_subdir_for_models}',
            f'_MODELS_SEED_DIR = _P(_REPO) / "models" / "{ds}"{seed_subdir_for_models}',
            f'TABLES_DIR = _P(_REPO) / "results" / "tables"{seed_subdir_for_tables}',
            'CALIB_OUT_DIR.mkdir(parents=True, exist_ok=True)',
            'TABLES_DIR.mkdir(parents=True, exist_ok=True)',
            'try:',
            '    PROBS_DIR = _MODELS_SEED_DIR / "probabilities"',
            '    PREDS_DIR = _MODELS_SEED_DIR / "predictions"',
            'except Exception: pass',
        ])
    elif notebook_name == '03e_refit_hybrid_calibrators.ipynb':
        if new_seed == 42:
            lines.extend([
                'for _ds in ["nsl_kdd_v2", "unsw_nb15_v2", "cic_ids2017_v2"]:',
                '    (_P(_REPO) / "calibrators" / _ds).mkdir(parents=True, exist_ok=True)',
            ])
        else:
            lines.extend([
                'for _ds in ["nsl_kdd_v2", "unsw_nb15_v2", "cic_ids2017_v2"]:',
                '    (_P(_REPO) / "calibrators" / _ds / _SEED_TAG).mkdir(parents=True, exist_ok=True)',
            ])
    elif notebook_name == '04c_shap_canonical.ipynb':
        if new_seed == 42:
            lines.extend([
                'for _ds in ["nsl_kdd_v2", "unsw_nb15_v2", "cic_ids2017_v2"]:',
                '    (_P(_REPO) / "shap_values" / _ds).mkdir(parents=True, exist_ok=True)',
                'TABLES_DIR = _P(_REPO) / "results" / "tables"',
                'TABLES_DIR.mkdir(parents=True, exist_ok=True)',
            ])
        else:
            lines.extend([
                'for _ds in ["nsl_kdd_v2", "unsw_nb15_v2", "cic_ids2017_v2"]:',
                '    (_P(_REPO) / "shap_values" / _ds / _SEED_TAG).mkdir(parents=True, exist_ok=True)',
                'TABLES_DIR = _P(_REPO) / "results" / "tables" / _SEED_TAG',
                'TABLES_DIR.mkdir(parents=True, exist_ok=True)',
            ])
    else:
        if new_seed == 42:
            lines.extend([
                'TABLES_DIR = _P(_REPO) / "results" / "tables"',
                'FIGURES_DIR = _P(_REPO) / "results" / "figures"',
                'TABLES_DIR.mkdir(parents=True, exist_ok=True)',
                'FIGURES_DIR.mkdir(parents=True, exist_ok=True)',
                'for _ds in ["nsl_kdd_v2", "unsw_nb15_v2", "cic_ids2017_v2"]:',
                '    (_P(_REPO) / "calibrators" / _ds).mkdir(parents=True, exist_ok=True)',
            ])
        else:
            lines.extend([
                'TABLES_DIR = _P(_REPO) / "results" / "tables" / _SEED_TAG',
                'FIGURES_DIR = _P(_REPO) / "results" / "figures" / _SEED_TAG',
                'TABLES_DIR.mkdir(parents=True, exist_ok=True)',
                'FIGURES_DIR.mkdir(parents=True, exist_ok=True)',
                'for _ds in ["nsl_kdd_v2", "unsw_nb15_v2", "cic_ids2017_v2"]:',
                '    (_P(_REPO) / "calibrators" / _ds / _SEED_TAG).mkdir(parents=True, exist_ok=True)',
            ])

    lines.append('print(f"[multi-seed patch] SEED={SEED}, TABLES_DIR={TABLES_DIR}")')
    return '\n'.join(lines)

# NEW: fixed patch_notebook_for_seed with corrected injection point
def patch_notebook_for_seed(nb_path, notebook_name, dataset_tag, strategy, new_seed):
    nb_obj = nbf.read(str(nb_path), as_version=4)
    patch_fn = PATCH_FUNCTIONS[strategy]

    for cell in nb_obj.cells:
        if cell.cell_type == 'code':
            cell.source = patch_fn(cell.source, new_seed)

    override_code = build_path_override_cell(notebook_name, dataset_tag, new_seed)
    override_cell = nbf.v4.new_code_cell(override_code)

    # NEW: insert AFTER the last cell defining any path constant
    path_pattern = re.compile(
        r'^\s*(MODELS_DIR|PREDS_DIR|PROBS_DIR|TABLES_DIR|CALIB_OUT_DIR|FIGURES_DIR|SHAP_DIR)\s*=',
        re.MULTILINE
    )
    insertion_idx = None
    for idx, cell in enumerate(nb_obj.cells):
        if cell.cell_type != 'code':
            continue
        if path_pattern.search(cell.source):
            insertion_idx = idx + 1  # Insert AFTER this cell (keep scanning to find LAST)

    if insertion_idx is None:
        for idx, cell in enumerate(nb_obj.cells):
            if cell.cell_type != 'code':
                continue
            if ('SEED' in cell.source) or ('import numpy' in cell.source) or ('import pandas' in cell.source):
                insertion_idx = idx + 1
                break
    if insertion_idx is None:
        for idx, cell in enumerate(nb_obj.cells):
            if cell.cell_type == 'code':
                insertion_idx = idx + 1
                break

    if insertion_idx is not None:
        nb_obj.cells.insert(insertion_idx, override_cell)

    return nb_obj, insertion_idx

print('All functions redefined. PIPELINE has', len(PIPELINE), 'notebooks.')
print('build_path_override_cell now handles seed=42 specially (uses original paths).')
print('patch_notebook_for_seed now inserts override AFTER last path-defining cell.')

# Now run verification
print()
print('Verification of new injection logic:')
print('=' * 80)
nb_dir = Path(REPO) / 'notebooks'

all_ok = True
for nb_name, ds_tag, strategy in PIPELINE:
    src_path = nb_dir / nb_name
    nb_obj, insert_idx = patch_notebook_for_seed(src_path, nb_name, ds_tag, strategy, 123)

    path_pattern = re.compile(
        r'^\s*(MODELS_DIR|PREDS_DIR|PROBS_DIR|TABLES_DIR|CALIB_OUT_DIR)\s*=',
        re.MULTILINE
    )

    path_cells = []
    override_cells = []
    for ci, c in enumerate(nb_obj.cells):
        if c.cell_type != 'code':
            continue
        if 'MULTI-SEED PATCH' in c.source:
            override_cells.append(ci)
        elif path_pattern.search(c.source):
            path_cells.append(ci)

    last_path_cell = max(path_cells) if path_cells else None
    first_override_cell = min(override_cells) if override_cells else None

    if last_path_cell is None:
        verdict = 'no path cells (notebook has no path constants — OK)'
        ok = True
    elif first_override_cell is None:
        verdict = 'OVERRIDE CELL NOT FOUND'
        ok = False
    elif first_override_cell > last_path_cell:
        verdict = f'OK: override@{first_override_cell} runs AFTER paths@{last_path_cell}'
        ok = True
    else:
        verdict = f'BUG: override@{first_override_cell} runs BEFORE paths@{last_path_cell}'
        ok = False

    if not ok:
        all_ok = False
    marker = 'OK' if ok else 'FAIL'
    print(f'  [{marker}] {nb_name:<42} {verdict}')

print()
print('=' * 80)
print(f'VERDICT: {"ALL OK — fix is correct" if all_ok else "STILL BUGGY — need to investigate"}')
print('=' * 80)

All functions redefined. PIPELINE has 15 notebooks.
build_path_override_cell now handles seed=42 specially (uses original paths).
patch_notebook_for_seed now inserts override AFTER last path-defining cell.

Verification of new injection logic:
  [OK] 02_train_models_v2.ipynb                   OK: override@5 runs AFTER paths@4
  [OK] 02_unsw_train_models_v2.ipynb              OK: override@6 runs AFTER paths@5
  [OK] 02_cic_train_models_v2.ipynb               OK: override@6 runs AFTER paths@5
  [OK] 03_nsl_calibration_v2.ipynb                OK: override@21 runs AFTER paths@20
  [OK] 03_unsw_calibration_v2.ipynb               OK: override@12 runs AFTER paths@11
  [OK] 03_cic_calibration_v2.ipynb                OK: override@20 runs AFTER paths@19
  [OK] 03e_refit_hybrid_calibrators.ipynb         no path cells (notebook has no path constants — OK)
  [OK] 04c_shap_canonical.ipynb                   no path cells (notebook has no path constants — OK)
  [OK] 05c_stability_canonical.ipynb      

In [14]:
import nbformat as nbf
from pathlib import Path

REPO = '/content/drive/MyDrive/XIDS_Research/xids-research'

# Inspect how 04c_shap_canonical loads/saves files
nb = nbf.read(str(Path(REPO) / 'notebooks' / '04c_shap_canonical.ipynb'), as_version=4)
print('=== 04c_shap_canonical.ipynb: file I/O lines ===')
for ci, c in enumerate(nb.cells):
    if c.cell_type != 'code': continue
    for li, line in enumerate(c.source.split('\n')):
        if any(s in line for s in ['np.save(', 'np.load(', 'joblib.', '.to_csv', 'Path(REPO)', "'models/", '"models/', 'shap_values', 'calibrators/']):
            stripped = line.strip()
            if stripped and not stripped.startswith('#'):
                print(f'  cell {ci} line {li}: {stripped[:150]}')

=== 04c_shap_canonical.ipynb: file I/O lines ===
  cell 5 line 16: base = Path(REPO) / 'models' / dataset
  cell 5 line 24: p = Path(REPO) / 'models' / dataset / f'{model_name}.pt'
  cell 7 line 4: PROCESSED = Path(REPO) / 'data' / 'processed' / ds
  cell 7 line 5: y_test = np.load(PROCESSED / 'y_test_5class.npy')
  cell 7 line 6: y_calib = np.load(PROCESSED / 'y_calib_5class.npy')
  cell 7 line 26: out_dir = Path(REPO) / 'shap_values' / ds
  cell 7 line 28: np.save(out_dir / 'canonical_eval_idx.npy', indices['eval_idx'])
  cell 7 line 29: np.save(out_dir / 'canonical_bg_idx.npy', indices['bg_idx'])
  cell 7 line 30: print('\n✓ Canonical indices saved to shap_values/{ds}/canonical_*.npy')
  cell 9 line 6: PROCESSED = Path(REPO) / 'data' / 'processed' / dataset
  cell 9 line 7: X_test = np.load(PROCESSED / 'X_test.npy').astype(np.float32)
  cell 9 line 8: X_calib = np.load(PROCESSED / 'X_calib.npy').astype(np.float32)
  cell 9 line 20: model = joblib.load(path)
  cell 9 line 22: shap_va

In [ ]:
# Save run log and produce summary
df_log = pd.DataFrame(run_log)
log_dir = Path(REPO) / 'models' / 'seed_runs' / '_run_logs'
log_dir.mkdir(parents=True, exist_ok=True)
log_csv = log_dir / f'run_log_{datetime.now().strftime("%Y%m%d_%H%M")}.csv'
df_log.to_csv(log_csv, index=False)
print(f'Saved run log: {log_csv}')

print()
print('=' * 70)
print('RUN SUMMARY')
print('=' * 70)
if len(df_log) > 0:
    status_counts = df_log['status'].value_counts()
    for k, v in status_counts.items():
        print(f'  {k}: {v}')

    print()
    print('Per-seed breakdown:')
    for seed in df_log['seed'].unique():
        sub = df_log[df_log['seed'] == seed]
        succ = (sub['status'] == 'success').sum()
        skip = (sub['status'] == 'skipped').sum()
        fail = (sub['status'] == 'failed').sum()
        halt = (sub['status'] == 'halted').sum()
        dry = (sub['status'] == 'dry_run').sum()
        elapsed = sub['elapsed_sec'].sum()
        print(f'  seed={seed}: success={succ}, skipped={skip}, failed={fail}, halted={halt}, dry_run={dry}, total_time={elapsed:.0f}s')

# If any failures, list them
fails = df_log[df_log['status'] == 'failed']
if len(fails) > 0:
    print()
    print('FAILURES:')
    for _, row in fails.iterrows():
        print(f'  seed={row["seed"]} | {row["notebook"]} | {row.get("error", "")[:120]}')

In [ ]:
from pathlib import Path
REPO = '/content/drive/MyDrive/XIDS_Research/xids-research'

for seed in [123, 456, 789]:
    base = Path(REPO) / 'models' / 'seed_runs' / f'seed{seed}'
    executed = base / '_executed'
    if executed.exists():
        files = list(executed.glob('*.ipynb'))
        print(f'seed{seed}/_executed: {len(files)} notebooks')
        for f in sorted(files):
            print(f'  {f.name} ({f.stat().st_size:,} bytes)')
    else:
        print(f'seed{seed}/_executed: NOT CREATED')

In [ ]:
# Quick verification: do seeds 2024 and 31337 patch correctly?
import sys
sys.path.insert(0, '/content/drive/MyDrive/XIDS_Research/xids-research/notebooks')

# We don't need to re-execute the dry-run notebook. We can just re-run the verification logic.
import nbformat as nbf
import re
from pathlib import Path

REPO = '/content/drive/MyDrive/XIDS_Research/xids-research'
nb_dir = Path(REPO) / 'notebooks'

# Patch primitives — same as runner
def patch_simple_seed(source, new_seed):
    pattern = re.compile(r'^(\s*SEED\s*=\s*)42(\s*(?:#.*)?)$', re.MULTILINE)
    return pattern.sub(lambda m: f'{m.group(1)}{new_seed}{m.group(2)}', source)

def patch_inject_seed_and_regex(source, new_seed):
    if re.search(r'^\s*SEED\s*=\s*\d+', source, re.MULTILINE):
        source = patch_simple_seed(source, new_seed)
    return re.sub(r'random_state\s*=\s*42\b', 'random_state=SEED', source)

def patch_no_seed(source, new_seed):
    return source

PATCH_FUNCTIONS = {
    'simple_seed': patch_simple_seed,
    'inject_seed_and_regex': patch_inject_seed_and_regex,
    'simple_seed_keep_boot': patch_simple_seed,
    'simple_seed_keep_defaults': patch_simple_seed,
    'no_seed': patch_no_seed,
}

PIPELINE = [
    ('02_train_models_v2.ipynb', 'simple_seed'),
    ('02_unsw_train_models_v2.ipynb', 'inject_seed_and_regex'),
    ('02_cic_train_models_v2.ipynb', 'inject_seed_and_regex'),
    ('03_nsl_calibration_v2.ipynb', 'simple_seed'),
    ('03_unsw_calibration_v2.ipynb', 'simple_seed'),
    ('03_cic_calibration_v2.ipynb', 'simple_seed'),
    ('03e_refit_hybrid_calibrators.ipynb', 'no_seed'),
    ('04c_shap_canonical.ipynb', 'simple_seed'),
    ('05c_stability_canonical.ipynb', 'simple_seed'),
    ('06_krishna_agreement_v3.ipynb', 'simple_seed'),
    ('07c_scts_canonical_mondrian_v2.ipynb', 'simple_seed'),
    ('07d_scts_calib_health.ipynb', 'no_seed'),
    ('07e_phase_a_strict_protocol.ipynb', 'simple_seed_keep_boot'),
    ('07f_phase_a_diagnostic.ipynb', 'simple_seed'),
    ('08_bootstrap_cis.ipynb', 'simple_seed_keep_defaults'),
]

all_pass = True
for seed in [2024, 31337]:
    print(f'\n=== Seed {seed} verification ===')
    for nb_name, strategy in PIPELINE:
        nb_obj = nbf.read(str(nb_dir / nb_name), as_version=4)
        patch_fn = PATCH_FUNCTIONS[strategy]
        all_text = []
        for cell in nb_obj.cells:
            if cell.cell_type == 'code':
                all_text.append(patch_fn(cell.source, seed))
        full_text = '\n\n'.join(all_text)

        # Check 1: no leftover random_state=42
        leftover_42 = len(re.findall(r'random_state\s*=\s*42\b', full_text))
        # Check 2: SEED assignments are correct
        seed_assigns = re.findall(r'^\s*SEED\s*=\s*(\d+)', full_text, re.MULTILINE)
        seed_correct = all(int(v) == seed for v in seed_assigns) if seed_assigns else (strategy == 'no_seed')

        # Check 3: BOOTSTRAP_SEED preserved in 07e
        boot_check = True
        if nb_name == '07e_phase_a_strict_protocol.ipynb':
            boot_check = bool(re.search(r'^\s*BOOTSTRAP_SEED\s*=\s*42', full_text, re.MULTILINE))
        # Check 4: function defaults preserved in 08
        defaults_check = True
        if nb_name == '08_bootstrap_cis.ipynb':
            defaults_check = bool(re.search(r'def\s+\w+\([^)]*\bseed\s*=\s*42', full_text))

        ok = (leftover_42 == 0) and seed_correct and boot_check and defaults_check
        if not ok:
            all_pass = False
            print(f'  FAIL: {nb_name} (leftover_42={leftover_42}, seed_correct={seed_correct}, boot={boot_check}, defaults={defaults_check})')
        else:
            print(f'  OK:   {nb_name}')

print(f'\n{"=" * 50}')
print(f'VERDICT: {"ALL PASS — ready for 5-seed launch" if all_pass else "FAIL — investigate before launching"}')
print(f'{"=" * 50}')

In [ ]:
from pathlib import Path
import os

REPO = '/content/drive/MyDrive/XIDS_Research/xids-research'
nb_dir = Path(REPO) / 'notebooks'

# Test 1: is the directory even visible?
print(f'Notebooks dir exists: {nb_dir.exists()}')
print(f'REPO dir exists: {Path(REPO).exists()}')

# Test 2: can we list contents?
if nb_dir.exists():
    files = sorted(nb_dir.glob('02_*.ipynb'))
    print(f'\n02_*.ipynb files visible: {len(files)}')
    for f in files[:5]:
        print(f'  {f.name}')

# Test 3: specifically check the file that errored
target = nb_dir / '02_train_models_v2.ipynb'
print(f'\nTarget file: {target}')
print(f'  exists(): {target.exists()}')
print(f'  is_file(): {target.is_file()}')

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import nbformat as nbf
import re
from pathlib import Path

REPO = '/content/drive/MyDrive/XIDS_Research/xids-research'
nb_dir = Path(REPO) / 'notebooks'

# Patch primitives — same as runner
def patch_simple_seed(source, new_seed):
    pattern = re.compile(r'^(\s*SEED\s*=\s*)42(\s*(?:#.*)?)$', re.MULTILINE)
    return pattern.sub(lambda m: f'{m.group(1)}{new_seed}{m.group(2)}', source)

def patch_inject_seed_and_regex(source, new_seed):
    if re.search(r'^\s*SEED\s*=\s*\d+', source, re.MULTILINE):
        source = patch_simple_seed(source, new_seed)
    return re.sub(r'random_state\s*=\s*42\b', 'random_state=SEED', source)

def patch_no_seed(source, new_seed):
    return source

PATCH_FUNCTIONS = {
    'simple_seed': patch_simple_seed,
    'inject_seed_and_regex': patch_inject_seed_and_regex,
    'simple_seed_keep_boot': patch_simple_seed,
    'simple_seed_keep_defaults': patch_simple_seed,
    'no_seed': patch_no_seed,
}

def build_override_cell_source(new_seed):
    """The override cell injects SEED via try/except NameError."""
    return f'''try:
    SEED
except NameError:
    SEED = {new_seed}
'''

PIPELINE = [
    ('02_train_models_v2.ipynb', 'simple_seed'),
    ('02_unsw_train_models_v2.ipynb', 'inject_seed_and_regex'),
    ('02_cic_train_models_v2.ipynb', 'inject_seed_and_regex'),
    ('03_nsl_calibration_v2.ipynb', 'simple_seed'),
    ('03_unsw_calibration_v2.ipynb', 'simple_seed'),
    ('03_cic_calibration_v2.ipynb', 'simple_seed'),
    ('03e_refit_hybrid_calibrators.ipynb', 'no_seed'),
    ('04c_shap_canonical.ipynb', 'simple_seed'),
    ('05c_stability_canonical.ipynb', 'simple_seed'),
    ('06_krishna_agreement_v3.ipynb', 'simple_seed'),
    ('07c_scts_canonical_mondrian_v2.ipynb', 'simple_seed'),
    ('07d_scts_calib_health.ipynb', 'no_seed'),
    ('07e_phase_a_strict_protocol.ipynb', 'simple_seed_keep_boot'),
    ('07f_phase_a_diagnostic.ipynb', 'simple_seed'),
    ('08_bootstrap_cis.ipynb', 'simple_seed_keep_defaults'),
]

all_pass = True
for seed in [2024, 31337]:
    print(f'\n=== Seed {seed} verification (with override cell included) ===')
    override_source = build_override_cell_source(seed)

    for nb_name, strategy in PIPELINE:
        nb_obj = nbf.read(str(nb_dir / nb_name), as_version=4)
        patch_fn = PATCH_FUNCTIONS[strategy]
        all_text = [override_source]  # Include override cell as if injected
        for cell in nb_obj.cells:
            if cell.cell_type == 'code':
                all_text.append(patch_fn(cell.source, seed))
        full_text = '\n\n'.join(all_text)

        # Check 1: no leftover random_state=42
        leftover_42 = len(re.findall(r'random_state\s*=\s*42\b', full_text))
        # Check 2: SEED assignments — now should include the override cell's
        seed_assigns = re.findall(r'^\s*SEED\s*=\s*(\d+)', full_text, re.MULTILINE)
        seed_correct = len(seed_assigns) > 0 and all(int(v) == seed for v in seed_assigns)
        if strategy == 'no_seed':
            seed_correct = True  # Override cell provides SEED for these too
        # Check 3: BOOTSTRAP_SEED preserved
        boot_check = True
        if nb_name == '07e_phase_a_strict_protocol.ipynb':
            boot_check = bool(re.search(r'^\s*BOOTSTRAP_SEED\s*=\s*42', full_text, re.MULTILINE))
        # Check 4: function defaults preserved
        defaults_check = True
        if nb_name == '08_bootstrap_cis.ipynb':
            defaults_check = bool(re.search(r'def\s+\w+\([^)]*\bseed\s*=\s*42', full_text))

        ok = (leftover_42 == 0) and seed_correct and boot_check and defaults_check
        if not ok:
            all_pass = False
            print(f'  FAIL: {nb_name} (leftover_42={leftover_42}, seed_correct={seed_correct} {seed_assigns}, boot={boot_check}, defaults={defaults_check})')
        else:
            print(f'  OK:   {nb_name} (SEED assignments: {seed_assigns})')

print(f'\n{"=" * 50}')
print(f'VERDICT: {"ALL PASS — ready for 5-seed launch" if all_pass else "FAIL — investigate"}')
print(f'{"=" * 50}')


=== Seed 2024 verification (with override cell included) ===
  OK:   02_train_models_v2.ipynb (SEED assignments: ['2024', '2024'])
  OK:   02_unsw_train_models_v2.ipynb (SEED assignments: ['2024'])
  OK:   02_cic_train_models_v2.ipynb (SEED assignments: ['2024'])
  OK:   03_nsl_calibration_v2.ipynb (SEED assignments: ['2024', '2024'])
  OK:   03_unsw_calibration_v2.ipynb (SEED assignments: ['2024', '2024'])
  OK:   03_cic_calibration_v2.ipynb (SEED assignments: ['2024', '2024'])
  OK:   03e_refit_hybrid_calibrators.ipynb (SEED assignments: ['2024'])
  OK:   04c_shap_canonical.ipynb (SEED assignments: ['2024', '2024'])
  OK:   05c_stability_canonical.ipynb (SEED assignments: ['2024', '2024'])
  OK:   06_krishna_agreement_v3.ipynb (SEED assignments: ['2024', '2024'])
  OK:   07c_scts_canonical_mondrian_v2.ipynb (SEED assignments: ['2024', '2024'])
  OK:   07d_scts_calib_health.ipynb (SEED assignments: ['2024'])
  OK:   07e_phase_a_strict_protocol.ipynb (SEED assignments: ['2024', '2024'

## Workflow

### First run (verify the runner is sane)

1. Keep `DRY_RUN_ONLY = True` (default in cell 2)
2. Run all cells
3. Look at the output of cell 8 (pre-flight) and cell 9 (per-notebook patch + would-execute log)
4. Open `models/seed_runs/seed123/_executed/02_train_models_v2.ipynb` and spot-check — should be the patched version

If everything looks right:

### Real execution

1. Edit cell 2: set `DRY_RUN_ONLY = False`
2. Re-run from cell 2 onwards
3. The runner will start actually executing notebooks
4. Wallclock estimate: ~2.5-3 hours per seed
5. Colab Pro sessions disconnect after ~6-8 hours of inactivity — long-running computation usually survives

### If Colab disconnects mid-run

1. Reopen the notebook
2. Re-run all cells from the top
3. The runner will skip notebooks whose checkpoint files exist
4. It resumes from where it left off

### If a notebook fails

1. Check `models/seed_runs/seed{N}/_errors/{notebook}.log` for the traceback
2. Open `models/seed_runs/seed{N}/_executed/{notebook}_failed.ipynb` to see the partial run
3. Fix the underlying issue (might require editing the original notebook)
4. Delete the failed-checkpoint files if any
5. Rerun

### After all seeds complete

Run the aggregator notebook (`00_multi_seed_aggregator.ipynb` — TBD).
This will load seed-42 + seed-123 + seed-456 + seed-789 results and produce
across-seeds median + IQR for all headline metrics.

### Commit discipline

After each successful seed completes (i.e., all 15 notebooks done), commit
the seed-N outputs to git:

```python
import os
os.chdir(REPO)
!git add models/seed_runs/seed{N}/
!git add models/{datasets}/seed{N}/
!git add calibrators/{datasets}/seed{N}/
!git add shap_values/{datasets}/seed{N}/
!git add results/tables/seed{N}/
!git add results/figures/seed{N}/
!git commit -m "Block 2 multi-seed: seed={N} pipeline complete (15 notebooks)"
!git push origin main
```

Replace `{N}` with 123, 456, or 789. We'll commit per-seed not per-notebook
to avoid 45 small commits.